# FRTB Sensitivity-Based Method (SBM) Capital Calculation

This notebook demonstrates the **FRTB** (Fundamental Review of the Trading Book, Basel IV)
regulatory capital calculation using the **Sensitivity-Based Method**, parallelised with
**ParFun** (data-level trade distribution) and **ParGraph** (DAG-level task scheduling).

The SBM computes market risk capital across **five risk classes**: Interest Rate (GIRR),
Credit Spread (CSR), Equity (EQ), Foreign Exchange (FX), and Commodity (CMDTY).
Within-bucket capital is:

$$
K_b = \sqrt{\sum_i W\!s_i^2 + \sum_{i \neq j} \rho_{ij}\, W\!s_i\, W\!s_j}\; +\; \text{Curvature}
$$

The pipeline has **5 levels**:

1. **Load and enrich trades** — generate a multi-asset trading book with market data
2. **Sensitivity computation** — bump-and-reprice across 5 risk classes (GIRR, CSR, EQ, FX, CMDTY), each with yield-curve bootstraps, Monte Carlo repricing, and PLAT back-tests
3. **Bucket capital under 3 correlation scenarios** — compute within-bucket and across-bucket Kb/Ks under low/medium/high intra-bucket correlation (15 nodes)
4. **Worst-case selection** — take the maximum capital across the 3 scenarios per risk class (regulatory conservative approach)
5. **Cross-class aggregation** — aggregate using the regulatory FRTB gamma matrix:

$$
\text{Total}_{\text{SBM}} = \sqrt{\sum_{b,c} \gamma_{bc}\, K\!s_b\, K\!s_c}
$$

**ParFun** parallelises the compute-heavy sensitivity phase by distributing all trades
in a single flat `@pf.parallel` batch across Scaler workers. **ParGraph** schedules
the full pipeline as a DAG: 5 sensitivity nodes fire in parallel, each spawning 3
correlation scenario nodes, converging through worst-case selection to the final aggregation.

## Program Structure

![Program Structure](FRTB_flowchart.svg)

# Native Python Implementation

In [ ]:
import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["GOTO_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import math
import random
import struct
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd


# ─── Tunable parameters ─────────────────────────────────────────────────────

N_TRADES = 100
EQ_MC_PATHS = 100
BOOTSTRAP_ITERS = 1000


# ─── Regulatory parameters (Basel IV FRTB CRE52) ────────────────────────────

GIRR_TENORS = [0.25, 0.5, 1, 2, 3, 5, 10, 15, 20, 30]

EQ_BUCKETS = {
    1: "Large-cap EM",
    2: "Large-cap DM",
    3: "Small-cap EM",
    4: "Small-cap DM",
    5: "Indices ETF",
    6: "Volatility",
}

CROSS_CLASS_CORR = np.array(
    [
        [1.00, 0.01, 0.04, 0.04, 0.04],
        [0.01, 1.00, 0.05, 0.04, 0.08],
        [0.04, 0.05, 1.00, 0.15, 0.20],
        [0.04, 0.04, 0.15, 1.00, 0.08],
        [0.04, 0.08, 0.20, 0.08, 1.00],
    ]
)

CORR_SCENARIOS = {
    "low": 0.75,
    "medium": 1.00,
    "high": 1.25,
}


# ─── Domain types ────────────────────────────────────────────────────────────


@dataclass
class Trade:
    trade_id: str
    asset_class: str
    instrument: str
    notional: float
    currency: str
    maturity: float
    fixed_rate: float
    ticker: str
    bucket: int
    spot: float
    vol: float
    credit_spread: float


@dataclass
class Sensitivity:
    trade_id: str
    risk_class: str
    bucket: int
    risk_factor: str
    delta: float
    vega: float
    curvature_up: float
    curvature_dn: float


@dataclass
class BucketedSensitivities:
    risk_class: str
    by_bucket: Dict[int, List[Sensitivity]]
    n_trades: int


@dataclass
class BucketCapital:
    risk_class: str
    corr_scenario: str
    kb_per_bucket: Dict[int, float]
    ks: float


@dataclass
class RiskClassCapital:
    risk_class: str
    capital: float
    worst_scenario: str
    scenario_capitals: Dict[str, float]


@dataclass
class FRTBCapitalReport:
    total_capital: float
    by_risk_class: Dict[str, float]
    scenario_breakdown: pd.DataFrame
    sensitivity_summary: pd.DataFrame


# ─── Pricing functions ───────────────────────────────────────────────────────


def _price_ir_swap(notional: float, fixed_rate: float, maturity: float, flat_rate: float) -> float:
    T = maturity
    annuity = sum(np.exp(-flat_rate * t) for t in np.arange(1, T + 1))
    par_rate = (1 - np.exp(-flat_rate * T)) / annuity
    return notional * (par_rate - fixed_rate) * annuity


def _price_equity(
    notional: float,
    spot: float,
    vol: float,
    maturity: float,
    strike: float,
    rate: float,
    flag: str = "call",
    n_paths: int = 5_000,
) -> float:
    seed_bytes = struct.pack("ddddd", spot, vol, maturity, strike, rate)
    local_rng = random.Random(seed_bytes)

    if maturity <= 0 or vol <= 0:
        return 0.0
    dt = maturity / 252.0
    n_steps = max(int(maturity * 252), 1)

    payoff_sum = 0.0
    drift = (rate - 0.5 * vol**2) * dt
    v_sqrt_dt = vol * math.sqrt(dt)

    for i in range(n_paths):
        path_ret = 0.0
        for j in range(n_steps):
            z = local_rng.gauss(0.0, 1.0)
            path_ret += drift + v_sqrt_dt * z

        ST = spot * math.exp(path_ret)
        if flag == "call":
            payoff = max(ST - strike, 0.0)
        else:
            payoff = max(strike - ST, 0.0)
        payoff_sum += payoff

    return notional * math.exp(-rate * maturity) * (payoff_sum / n_paths)


def _price_credit_bond(notional: float, coupon: float, maturity: float, risk_free: float, spread: float) -> float:
    ytm = risk_free + spread
    T = int(max(maturity, 1))
    pv_coupons = sum(notional * coupon * np.exp(-ytm * t) for t in range(1, T + 1))
    pv_principal = notional * np.exp(-ytm * maturity)
    return pv_coupons + pv_principal


def _bootstrap_yield_curve(base_rate: float, n_instruments: int = 30, n_iters: int = 5) -> np.ndarray:
    seed_bytes = struct.pack("di", base_rate, n_instruments)
    local_rng = random.Random(seed_bytes)
    A = np.zeros((n_instruments, n_instruments))
    for i in range(n_instruments):
        for j in range(n_instruments):
            val = 0.0
            for k in range(n_iters):
                val += local_rng.gauss(0.0, 1.0)
            A[i, j] = val / max(n_iters, 1)

    A = A @ A.T + np.eye(n_instruments) * base_rate

    b = np.empty(n_instruments)
    for i in range(n_instruments):
        val = 0.0
        for k in range(n_iters):
            val += local_rng.gauss(0.0, 1.0)
        b[i] = val / max(n_iters, 1)

    L = np.linalg.cholesky(A)
    y = np.linalg.solve(L, b)
    discount_factors = np.linalg.solve(L.T, y)
    return discount_factors


def _run_historical_pnl_explain(sensitivities_list, rng, n_scenarios: int = 250) -> float:
    if not sensitivities_list:
        return 0.0
    n_sens = len(sensitivities_list)
    rf_moves = rng.standard_normal((n_scenarios, n_sens)) * 0.001
    ws_vec = np.array([s.delta for s in sensitivities_list])
    rtpl = rf_moves @ ws_vec
    hpnl = rtpl + rng.standard_normal(n_scenarios) * abs(ws_vec).mean() * 0.05
    residuals = hpnl - rtpl
    resid_matrix = np.outer(residuals, residuals) / n_scenarios
    _ = np.linalg.svd(resid_matrix)
    return float(np.sqrt(np.mean(residuals**2)))


# ─── Trade generation ────────────────────────────────────────────────────────


def load_and_enrich_trades(n_trades: int, seed: int = 42) -> List[Trade]:
    rng = np.random.default_rng(seed)
    asset_classes = ["GIRR", "CSR", "EQ", "FX", "CMDTY"]
    weights = [0.35, 0.25, 0.20, 0.12, 0.08]
    currencies = ["USD", "EUR", "GBP", "JPY", "CHF"]
    tickers = [f"TICKER_{i:04d}" for i in range(100)]

    trades = []
    for i in range(n_trades):
        ac = rng.choice(asset_classes, p=weights)
        trades.append(
            Trade(
                trade_id=f"TRD_{i:06d}",
                asset_class=ac,
                instrument={
                    "GIRR": "ir_swap",
                    "CSR": "bond",
                    "EQ": "equity_option",
                    "FX": "fx_option",
                    "CMDTY": "commodity_fwd",
                }[ac],
                notional=float(rng.choice([1e5, 5e5, 1e6, 5e6, 10e6])),
                currency=rng.choice(currencies),
                maturity=float(rng.uniform(0.5, 30.0)),
                fixed_rate=float(rng.uniform(0.02, 0.07)),
                ticker=rng.choice(tickers),
                bucket=int(rng.integers(1, 7)),
                spot=float(rng.uniform(50, 200)),
                vol=float(rng.uniform(0.10, 0.60)),
                credit_spread=float(rng.uniform(0.005, 0.025)),
            )
        )
    return trades


# ─── Per-trade sensitivity functions ─────────────────────────────────────────


def _compute_girr_sensitivities_for_trade(
    trade: Trade, base_rate: float, bump_size_ir: float, n_historical_scenarios: int, bootstrap_iters: int = 5
) -> Tuple[int, List[Sensitivity]]:
    rng = np.random.default_rng(int(trade.trade_id.split("_")[1]))
    RW_GIRR = {0.25: 1.74, 0.5: 1.74, 1: 0.74, 2: 0.58, 3: 0.49, 5: 0.44, 10: 0.40, 15: 0.39, 20: 0.38, 30: 0.38}

    _bootstrap_yield_curve(base_rate, n_instruments=30, n_iters=bootstrap_iters)
    base_npv = _price_ir_swap(trade.notional, trade.fixed_rate, trade.maturity, base_rate)

    trade_sens_list = []
    for tenor, rw in RW_GIRR.items():
        bumped_rate = base_rate + bump_size_ir * (tenor / max(trade.maturity, 0.25))
        _bootstrap_yield_curve(bumped_rate, n_instruments=30, n_iters=bootstrap_iters)
        bumped_npv = _price_ir_swap(trade.notional, trade.fixed_rate, trade.maturity, bumped_rate)
        delta_raw = bumped_npv - base_npv
        Ws = delta_raw * rw / bump_size_ir * 1e-4

        _bootstrap_yield_curve(base_rate + 0.01, n_instruments=30, n_iters=bootstrap_iters)
        _bootstrap_yield_curve(base_rate - 0.01, n_instruments=30, n_iters=bootstrap_iters)
        npv_up = _price_ir_swap(trade.notional, trade.fixed_rate, trade.maturity, base_rate + 0.01)
        npv_dn = _price_ir_swap(trade.notional, trade.fixed_rate, trade.maturity, base_rate - 0.01)
        cvr_up = npv_up - base_npv - delta_raw * 100
        cvr_dn = npv_dn - base_npv + delta_raw * 100

        trade_sens_list.append(
            Sensitivity(
                trade_id=trade.trade_id,
                risk_class="GIRR",
                bucket=trade.bucket,
                risk_factor=f"IR_{tenor}Y",
                delta=Ws,
                vega=0.0,
                curvature_up=cvr_up,
                curvature_dn=cvr_dn,
            )
        )

    _run_historical_pnl_explain(trade_sens_list, rng, n_historical_scenarios)
    return (trade.bucket, trade_sens_list)


def _compute_eq_sensitivities_for_trade(
    trade: Trade,
    base_rate: float,
    bump_size_eq: float,
    bump_size_vol: float,
    n_mc_paths: int,
    n_historical_scenarios: int,
) -> Tuple[int, List[Sensitivity]]:
    rng = np.random.default_rng(int(trade.trade_id.split("_")[1]))
    RW_EQ = {1: 0.55, 2: 0.35, 3: 0.45, 4: 0.30, 5: 0.20, 6: 0.70}
    strike = trade.spot * rng.uniform(0.9, 1.1)
    rw = RW_EQ.get(trade.bucket, 0.35)

    spot_bumps = [trade.spot * (1 + k * bump_size_eq) for k in range(-4, 6)]
    npvs = [
        _price_equity(trade.notional, s, trade.vol, trade.maturity, strike, base_rate, n_paths=n_mc_paths)
        for s in spot_bumps
    ]
    base_npv = npvs[4]
    delta_raw = npvs[5] - npvs[4]
    Ws_delta = delta_raw * rw / bump_size_eq

    vol_bumps = [trade.vol + k * bump_size_vol for k in range(-2, 3)]
    vol_npvs = [
        _price_equity(trade.notional, trade.spot, v, trade.maturity, strike, base_rate, n_paths=n_mc_paths)
        for v in vol_bumps
    ]
    vega_raw = vol_npvs[3] - vol_npvs[2]
    Ws_vega = vega_raw * 0.78 / bump_size_vol

    shock = rw * trade.spot
    npv_up = _price_equity(
        trade.notional, trade.spot + shock, trade.vol, trade.maturity, strike, base_rate, n_paths=n_mc_paths
    )
    npv_dn = _price_equity(
        trade.notional, trade.spot - shock, trade.vol, trade.maturity, strike, base_rate, n_paths=n_mc_paths
    )
    cvr_up = npv_up - base_npv - delta_raw * shock / (bump_size_eq * trade.spot)
    cvr_dn = npv_dn - base_npv + delta_raw * shock / (bump_size_eq * trade.spot)

    trade_sens = [
        Sensitivity(
            trade_id=trade.trade_id,
            risk_class="EQ",
            bucket=trade.bucket,
            risk_factor=f"EQ_B{trade.bucket}",
            delta=Ws_delta,
            vega=Ws_vega,
            curvature_up=cvr_up,
            curvature_dn=cvr_dn,
        )
    ]
    _run_historical_pnl_explain(trade_sens, rng, n_historical_scenarios)
    return (trade.bucket, trade_sens)


def _compute_csr_sensitivities_for_trade(
    trade: Trade, base_rate: float, bump_size_cs: float, n_historical_scenarios: int, bootstrap_iters: int = 5
) -> Tuple[int, List[Sensitivity]]:
    rng = np.random.default_rng(int(trade.trade_id.split("_")[1]))
    cs_tenors = [0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30]
    is_ig = trade.credit_spread < 0.01
    RW_CSR = 0.005 if is_ig else 0.05
    cs_shock = 0.06 if is_ig else 0.10

    trade_sens_list = []
    for cs_t in cs_tenors:
        if cs_t > trade.maturity + 1:
            continue

        base_npv = _price_credit_bond(trade.notional, trade.fixed_rate, trade.maturity, base_rate, trade.credit_spread)
        bump_spread = trade.credit_spread + bump_size_cs
        bumped_npv = _price_credit_bond(trade.notional, trade.fixed_rate, trade.maturity, base_rate, bump_spread)
        delta_raw = bumped_npv - base_npv
        Ws = delta_raw * RW_CSR / bump_size_cs

        npv_up = _price_credit_bond(
            trade.notional, trade.fixed_rate, trade.maturity, base_rate, trade.credit_spread + cs_shock
        )
        npv_dn = _price_credit_bond(
            trade.notional, trade.fixed_rate, trade.maturity, base_rate, max(trade.credit_spread - cs_shock, 0)
        )
        cvr_up = npv_up - base_npv - delta_raw * cs_shock / bump_size_cs
        cvr_dn = npv_dn - base_npv + delta_raw * cs_shock / bump_size_cs

        _bootstrap_yield_curve(base_rate + trade.credit_spread, n_instruments=30, n_iters=bootstrap_iters)

        trade_sens_list.append(
            Sensitivity(
                trade_id=trade.trade_id,
                risk_class="CSR",
                bucket=trade.bucket,
                risk_factor=f"CS_{cs_t}Y_{trade.currency}",
                delta=Ws,
                vega=0.0,
                curvature_up=cvr_up,
                curvature_dn=cvr_dn,
            )
        )

    _run_historical_pnl_explain(trade_sens_list, rng, n_historical_scenarios)
    return (trade.bucket, trade_sens_list)


def _compute_fx_sensitivities_for_trade(
    trade: Trade, base_rate: float, bump_size_eq: float, n_historical_scenarios: int, bootstrap_iters: int = 5
) -> Tuple[int, List[Sensitivity]]:
    rng = np.random.default_rng(int(trade.trade_id.split("_")[1]))
    vol_strikes = [0.10, 0.25, 0.50, 0.75, 0.90]
    fx_tenors = [1.0, 2.0]

    trade_sens_list = []
    for fx_t in fx_tenors:
        for vk in vol_strikes:
            base_npv = trade.notional * trade.spot
            bumped_npv = trade.notional * trade.spot * (1 + bump_size_eq)
            delta_raw = bumped_npv - base_npv
            Ws = delta_raw * 0.15 / bump_size_eq

            npv_up = trade.notional * trade.spot * 1.15
            npv_dn = trade.notional * trade.spot * 0.85
            cvr_up = npv_up - base_npv - delta_raw * 15
            cvr_dn = npv_dn - base_npv + delta_raw * 15

            _bootstrap_yield_curve(base_rate, n_instruments=15, n_iters=bootstrap_iters)

            trade_sens_list.append(
                Sensitivity(
                    trade_id=trade.trade_id,
                    risk_class="FX",
                    bucket=trade.bucket,
                    risk_factor=f"FX_{trade.currency}USD_T{fx_t}_K{vk}",
                    delta=Ws,
                    vega=0.0,
                    curvature_up=cvr_up,
                    curvature_dn=cvr_dn,
                )
            )

    _run_historical_pnl_explain(trade_sens_list, rng, n_historical_scenarios)
    return (trade.bucket, trade_sens_list)


def _compute_cmdty_sensitivities_for_trade(
    trade: Trade, base_rate: float, bump_size_eq: float, n_historical_scenarios: int, bootstrap_iters: int = 5
) -> Tuple[int, List[Sensitivity]]:
    rng = np.random.default_rng(int(trade.trade_id.split("_")[1]))
    RW_CMDTY = {1: 0.30, 2: 0.35, 3: 0.60, 4: 0.80, 5: 0.40, 6: 0.45, 7: 0.20, 8: 0.35, 9: 0.25}
    cmdty_tenors = [0.25, 0.5, 1, 2, 3, 5]
    grade_diffs = [-0.02, 0.0, 0.02]
    rw = RW_CMDTY.get(trade.bucket, 0.40)

    trade_sens_list = []
    for ct in cmdty_tenors:
        for gd in grade_diffs:
            spot_adj = trade.spot * (1 + gd)
            base_npv = trade.notional * spot_adj
            bumped_npv = trade.notional * spot_adj * (1 + bump_size_eq)
            delta_raw = bumped_npv - base_npv
            Ws = delta_raw * rw / bump_size_eq

            _bootstrap_yield_curve(base_rate + gd, n_instruments=20, n_iters=bootstrap_iters)

            trade_sens_list.append(
                Sensitivity(
                    trade_id=trade.trade_id,
                    risk_class="CMDTY",
                    bucket=trade.bucket,
                    risk_factor=f"CMDTY_B{trade.bucket}_T{ct}_G{gd:.2f}",
                    delta=Ws,
                    vega=0.0,
                    curvature_up=0.0,
                    curvature_dn=0.0,
                )
            )

    _run_historical_pnl_explain(trade_sens_list, rng, n_historical_scenarios)
    return (trade.bucket, trade_sens_list)


# ─── Unified per-trade dispatch ──────────────────────────────────────────────


def _compute_sensitivities_for_trade(
    trade: Trade,
    base_rate: float = 0.04,
    bump_size_ir: float = 0.0001,
    bump_size_eq: float = 0.01,
    bump_size_vol: float = 0.01,
    bump_size_cs: float = 0.0001,
    n_mc_paths: int = 100,
    n_historical_scenarios: int = 250,
    bootstrap_iters: int = 5,
) -> Tuple[str, int, List[Sensitivity]]:
    ac = trade.asset_class
    if ac == "GIRR":
        bid, sens = _compute_girr_sensitivities_for_trade(
            trade, base_rate, bump_size_ir, n_historical_scenarios, bootstrap_iters
        )
    elif ac == "EQ":
        bid, sens = _compute_eq_sensitivities_for_trade(
            trade, base_rate, bump_size_eq, bump_size_vol, n_mc_paths, n_historical_scenarios
        )
    elif ac == "CSR":
        bid, sens = _compute_csr_sensitivities_for_trade(
            trade, base_rate, bump_size_cs, n_historical_scenarios, bootstrap_iters
        )
    elif ac == "FX":
        bid, sens = _compute_fx_sensitivities_for_trade(
            trade, base_rate, bump_size_eq, n_historical_scenarios, bootstrap_iters
        )
    elif ac == "CMDTY":
        bid, sens = _compute_cmdty_sensitivities_for_trade(
            trade, base_rate, bump_size_eq, n_historical_scenarios, bootstrap_iters
        )
    else:
        raise ValueError(f"Unknown asset class: {ac}")
    return (ac, bid, sens)


def _group_sensitivities(results: List[Tuple[str, int, List[Sensitivity]]]) -> Dict[str, BucketedSensitivities]:
    by_class: Dict[str, Dict[int, List[Sensitivity]]] = {}
    trade_counts: Dict[str, int] = {}
    for risk_class, bucket_id, sens_list in results:
        if risk_class not in by_class:
            by_class[risk_class] = {}
            trade_counts[risk_class] = 0
        trade_counts[risk_class] += 1
        if bucket_id not in by_class[risk_class]:
            by_class[risk_class][bucket_id] = []
        by_class[risk_class][bucket_id].extend(sens_list)
    result = {}
    for rc in ["GIRR", "CSR", "EQ", "FX", "CMDTY"]:
        if rc in by_class:
            result[rc] = BucketedSensitivities(risk_class=rc, by_bucket=by_class[rc], n_trades=trade_counts[rc])
        else:
            result[rc] = BucketedSensitivities(risk_class=rc, by_bucket={}, n_trades=0)
    return result


# ─── Bucket capital computation ──────────────────────────────────────────────


def _compute_single_bucket_capital(
    bucket_id: int, sens_list: List[Sensitivity], rho_eff: float
) -> Tuple[int, float, float]:
    if not sens_list:
        return (bucket_id, 0.0, 0.0)

    Ws = np.array([s.delta for s in sens_list])
    n = len(Ws)

    corr_matrix = np.full((n, n), rho_eff)
    np.fill_diagonal(corr_matrix, 1.0)
    corr_matrix += np.eye(n) * 1e-8

    try:
        L = np.linalg.cholesky(corr_matrix)
        Lw = np.linalg.solve(L, Ws)
        Kb_sq = float(Lw @ Lw)
    except np.linalg.LinAlgError:
        Kb_sq = float(Ws @ corr_matrix @ Ws)

    cvr_up = np.array([s.curvature_up for s in sens_list])
    cvr_dn = np.array([s.curvature_dn for s in sens_list])
    curvature_capital = max(0.0, -float(np.minimum(cvr_up, cvr_dn).sum()))

    Kb = float(np.sqrt(max(Kb_sq, 0.0))) + curvature_capital
    Sb = float(Ws.sum())
    return (bucket_id, Kb, Sb)


def compute_bucket_capital_under_scenario(
    bucketed: BucketedSensitivities, corr_scenario: str
) -> BucketCapital:
    scenario_scalar = CORR_SCENARIOS[corr_scenario]
    risk_class = bucketed.risk_class

    INTRA_CORR = {"GIRR": 0.999, "CSR": 0.35, "EQ": 0.15, "FX": 0.60, "CMDTY": 0.20}
    rho_base = INTRA_CORR.get(risk_class, 0.30)
    rho_eff = min(max(rho_base * scenario_scalar, 0.0), 1.0)

    CROSS_BUCKET_CORR = {"GIRR": 0.0, "CSR": 0.0, "EQ": 0.15, "FX": 0.60, "CMDTY": 0.20}
    gamma = min(max(CROSS_BUCKET_CORR.get(risk_class, 0.0) * scenario_scalar, 0.0), 1.0)

    bucket_items = list(bucketed.by_bucket.items())
    results = [_compute_single_bucket_capital(bid, slist, rho_eff) for bid, slist in bucket_items]

    kb_per_bucket: Dict[int, float] = {bid: Kb for bid, Kb, _ in results}
    Sb_per_bucket: Dict[int, float] = {bid: Sb for bid, _, Sb in results}

    buckets = sorted(kb_per_bucket.keys())
    Kb_vec = np.array([kb_per_bucket[b] for b in buckets])
    Sb_vec = np.array([Sb_per_bucket[b] for b in buckets])

    n_b = len(buckets)
    if n_b == 0:
        Ks = 0.0
    elif n_b == 1:
        Ks = float(Kb_vec[0])
    else:
        cross_corr = np.full((n_b, n_b), gamma)
        np.fill_diagonal(cross_corr, 1.0)
        Ks_sq = float(Kb_vec @ Kb_vec) + float(Sb_vec @ cross_corr @ Sb_vec) - float(Sb_vec @ Sb_vec)
        Ks = float(np.sqrt(max(Ks_sq, 0.0)))

    return BucketCapital(risk_class=risk_class, corr_scenario=corr_scenario, kb_per_bucket=kb_per_bucket, ks=Ks)


def worst_case_risk_class_capital(low: BucketCapital, medium: BucketCapital, high: BucketCapital) -> RiskClassCapital:
    scenario_ks = {bc.corr_scenario: bc.ks for bc in [low, medium, high]}
    worst_scenario = max(scenario_ks, key=scenario_ks.get)
    return RiskClassCapital(
        risk_class=low.risk_class,
        capital=scenario_ks[worst_scenario],
        worst_scenario=worst_scenario,
        scenario_capitals=scenario_ks,
    )


def aggregate_total_capital(
    risk_class_capitals_girr: RiskClassCapital,
    risk_class_capitals_csr: RiskClassCapital,
    risk_class_capitals_eq: RiskClassCapital,
    risk_class_capitals_fx: RiskClassCapital,
    risk_class_capitals_cmdty: RiskClassCapital,
    trades: List[Trade],
) -> FRTBCapitalReport:
    ORDER = ["GIRR", "CSR", "EQ", "FX", "CMDTY"]
    rc_cap = {
        rc.risk_class: rc.capital
        for rc in [
            risk_class_capitals_girr,
            risk_class_capitals_csr,
            risk_class_capitals_eq,
            risk_class_capitals_fx,
            risk_class_capitals_cmdty,
        ]
    }

    Ks_vec = np.array([rc_cap.get(rc, 0.0) for rc in ORDER])
    total_sq = float(Ks_vec @ CROSS_CLASS_CORR @ Ks_vec)
    total_capital = float(np.sqrt(max(total_sq, 0.0)))

    by_risk_class = {rc: float(Ks_vec[i]) for i, rc in enumerate(ORDER)}

    rows = []
    for rc in [
        risk_class_capitals_girr,
        risk_class_capitals_csr,
        risk_class_capitals_eq,
        risk_class_capitals_fx,
        risk_class_capitals_cmdty,
    ]:
        for scenario, cap in rc.scenario_capitals.items():
            rows.append(
                {
                    "Risk Class": rc.risk_class,
                    "Scenario": scenario,
                    "Capital ($M)": cap / 1e6,
                    "Is Worst": scenario == rc.worst_scenario,
                }
            )
    scenario_df = pd.DataFrame(rows)

    asset_classes = [t.asset_class for t in trades]
    sens_df = pd.DataFrame({"Asset Class": asset_classes}).value_counts().reset_index()
    sens_df.columns = ["Asset Class", "Trade Count"]

    return FRTBCapitalReport(
        total_capital=total_capital,
        by_risk_class=by_risk_class,
        scenario_breakdown=scenario_df,
        sensitivity_summary=sens_df,
    )


# ─── Sequential pipeline ────────────────────────────────────────────────────


def frtb_sbm_capital(n_trades: int = N_TRADES, seed: int = 42, bootstrap_iters: int = BOOTSTRAP_ITERS) -> FRTBCapitalReport:
    trades = load_and_enrich_trades(n_trades=n_trades, seed=seed)

    # Compute all sensitivities sequentially
    all_results = [
        _compute_sensitivities_for_trade(t, n_mc_paths=EQ_MC_PATHS, bootstrap_iters=bootstrap_iters) for t in trades
    ]
    grouped = _group_sensitivities(all_results)

    # Post-sensitivity pipeline: scenarios → worst-case → aggregation
    RISK_CLASSES = ["GIRR", "CSR", "EQ", "FX", "CMDTY"]
    rc_capitals = {}
    for rc in RISK_CLASSES:
        bucketed = grouped[rc]
        caps = {}
        for scenario in ["low", "medium", "high"]:
            caps[scenario] = compute_bucket_capital_under_scenario(bucketed, scenario)
        rc_capitals[rc] = worst_case_risk_class_capital(caps["low"], caps["medium"], caps["high"])

    return aggregate_total_capital(
        rc_capitals["GIRR"], rc_capitals["CSR"], rc_capitals["EQ"], rc_capitals["FX"], rc_capitals["CMDTY"], trades
    )


t0 = time.perf_counter()
report = frtb_sbm_capital()
elapsed = time.perf_counter() - t0

print(f"Total FRTB SBM Capital: ${report.total_capital / 1e6:,.4f}M")
print(f"By risk class: { {k: f'${v/1e6:,.4f}M' for k, v in report.by_risk_class.items()} }")
print(f"Wall time: {elapsed:.1f}s")


# ParFun Implementation

In [ ]:
import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["GOTO_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import math
import random
import struct
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import parfun as pf
from scaler import SchedulerClusterCombo


# ─── Tunable parameters ─────────────────────────────────────────────────────

N_TRADES = 100
EQ_MC_PATHS = 100
BOOTSTRAP_ITERS = 1000


# ─── Regulatory parameters (Basel IV FRTB CRE52) ────────────────────────────

GIRR_TENORS = [0.25, 0.5, 1, 2, 3, 5, 10, 15, 20, 30]

EQ_BUCKETS = {
    1: "Large-cap EM",
    2: "Large-cap DM",
    3: "Small-cap EM",
    4: "Small-cap DM",
    5: "Indices ETF",
    6: "Volatility",
}

CROSS_CLASS_CORR = np.array(
    [
        [1.00, 0.01, 0.04, 0.04, 0.04],
        [0.01, 1.00, 0.05, 0.04, 0.08],
        [0.04, 0.05, 1.00, 0.15, 0.20],
        [0.04, 0.04, 0.15, 1.00, 0.08],
        [0.04, 0.08, 0.20, 0.08, 1.00],
    ]
)

CORR_SCENARIOS = {
    "low": 0.75,
    "medium": 1.00,
    "high": 1.25,
}


# ─── Domain types ────────────────────────────────────────────────────────────


@dataclass
class Trade:
    trade_id: str
    asset_class: str
    instrument: str
    notional: float
    currency: str
    maturity: float
    fixed_rate: float
    ticker: str
    bucket: int
    spot: float
    vol: float
    credit_spread: float


@dataclass
class Sensitivity:
    trade_id: str
    risk_class: str
    bucket: int
    risk_factor: str
    delta: float
    vega: float
    curvature_up: float
    curvature_dn: float


@dataclass
class BucketedSensitivities:
    risk_class: str
    by_bucket: Dict[int, List[Sensitivity]]
    n_trades: int


@dataclass
class BucketCapital:
    risk_class: str
    corr_scenario: str
    kb_per_bucket: Dict[int, float]
    ks: float


@dataclass
class RiskClassCapital:
    risk_class: str
    capital: float
    worst_scenario: str
    scenario_capitals: Dict[str, float]


@dataclass
class FRTBCapitalReport:
    total_capital: float
    by_risk_class: Dict[str, float]
    scenario_breakdown: pd.DataFrame
    sensitivity_summary: pd.DataFrame


# ─── Pricing functions ───────────────────────────────────────────────────────


def _price_ir_swap(notional: float, fixed_rate: float, maturity: float, flat_rate: float) -> float:
    T = maturity
    annuity = sum(np.exp(-flat_rate * t) for t in np.arange(1, T + 1))
    par_rate = (1 - np.exp(-flat_rate * T)) / annuity
    return notional * (par_rate - fixed_rate) * annuity


def _price_equity(
    notional: float,
    spot: float,
    vol: float,
    maturity: float,
    strike: float,
    rate: float,
    flag: str = "call",
    n_paths: int = 5_000,
) -> float:
    seed_bytes = struct.pack("ddddd", spot, vol, maturity, strike, rate)
    local_rng = random.Random(seed_bytes)

    if maturity <= 0 or vol <= 0:
        return 0.0
    dt = maturity / 252.0
    n_steps = max(int(maturity * 252), 1)

    payoff_sum = 0.0
    drift = (rate - 0.5 * vol**2) * dt
    v_sqrt_dt = vol * math.sqrt(dt)

    for i in range(n_paths):
        path_ret = 0.0
        for j in range(n_steps):
            z = local_rng.gauss(0.0, 1.0)
            path_ret += drift + v_sqrt_dt * z

        ST = spot * math.exp(path_ret)
        if flag == "call":
            payoff = max(ST - strike, 0.0)
        else:
            payoff = max(strike - ST, 0.0)
        payoff_sum += payoff

    return notional * math.exp(-rate * maturity) * (payoff_sum / n_paths)


def _price_credit_bond(notional: float, coupon: float, maturity: float, risk_free: float, spread: float) -> float:
    ytm = risk_free + spread
    T = int(max(maturity, 1))
    pv_coupons = sum(notional * coupon * np.exp(-ytm * t) for t in range(1, T + 1))
    pv_principal = notional * np.exp(-ytm * maturity)
    return pv_coupons + pv_principal


def _bootstrap_yield_curve(base_rate: float, n_instruments: int = 30, n_iters: int = 5) -> np.ndarray:
    seed_bytes = struct.pack("di", base_rate, n_instruments)
    local_rng = random.Random(seed_bytes)
    A = np.zeros((n_instruments, n_instruments))
    for i in range(n_instruments):
        for j in range(n_instruments):
            val = 0.0
            for k in range(n_iters):
                val += local_rng.gauss(0.0, 1.0)
            A[i, j] = val / max(n_iters, 1)

    A = A @ A.T + np.eye(n_instruments) * base_rate

    b = np.empty(n_instruments)
    for i in range(n_instruments):
        val = 0.0
        for k in range(n_iters):
            val += local_rng.gauss(0.0, 1.0)
        b[i] = val / max(n_iters, 1)

    L = np.linalg.cholesky(A)
    y = np.linalg.solve(L, b)
    discount_factors = np.linalg.solve(L.T, y)
    return discount_factors


def _run_historical_pnl_explain(sensitivities_list, rng, n_scenarios: int = 250) -> float:
    if not sensitivities_list:
        return 0.0
    n_sens = len(sensitivities_list)
    rf_moves = rng.standard_normal((n_scenarios, n_sens)) * 0.001
    ws_vec = np.array([s.delta for s in sensitivities_list])
    rtpl = rf_moves @ ws_vec
    hpnl = rtpl + rng.standard_normal(n_scenarios) * abs(ws_vec).mean() * 0.05
    residuals = hpnl - rtpl
    resid_matrix = np.outer(residuals, residuals) / n_scenarios
    _ = np.linalg.svd(resid_matrix)
    return float(np.sqrt(np.mean(residuals**2)))


# ─── Trade generation ────────────────────────────────────────────────────────


def load_and_enrich_trades(n_trades: int, seed: int = 42) -> List[Trade]:
    rng = np.random.default_rng(seed)
    asset_classes = ["GIRR", "CSR", "EQ", "FX", "CMDTY"]
    weights = [0.35, 0.25, 0.20, 0.12, 0.08]
    currencies = ["USD", "EUR", "GBP", "JPY", "CHF"]
    tickers = [f"TICKER_{i:04d}" for i in range(100)]

    trades = []
    for i in range(n_trades):
        ac = rng.choice(asset_classes, p=weights)
        trades.append(
            Trade(
                trade_id=f"TRD_{i:06d}",
                asset_class=ac,
                instrument={
                    "GIRR": "ir_swap",
                    "CSR": "bond",
                    "EQ": "equity_option",
                    "FX": "fx_option",
                    "CMDTY": "commodity_fwd",
                }[ac],
                notional=float(rng.choice([1e5, 5e5, 1e6, 5e6, 10e6])),
                currency=rng.choice(currencies),
                maturity=float(rng.uniform(0.5, 30.0)),
                fixed_rate=float(rng.uniform(0.02, 0.07)),
                ticker=rng.choice(tickers),
                bucket=int(rng.integers(1, 7)),
                spot=float(rng.uniform(50, 200)),
                vol=float(rng.uniform(0.10, 0.60)),
                credit_spread=float(rng.uniform(0.005, 0.025)),
            )
        )
    return trades


# ─── Per-trade sensitivity functions ─────────────────────────────────────────


def _compute_girr_sensitivities_for_trade(
    trade: Trade, base_rate: float, bump_size_ir: float, n_historical_scenarios: int, bootstrap_iters: int = 5
) -> Tuple[int, List[Sensitivity]]:
    rng = np.random.default_rng(int(trade.trade_id.split("_")[1]))
    RW_GIRR = {0.25: 1.74, 0.5: 1.74, 1: 0.74, 2: 0.58, 3: 0.49, 5: 0.44, 10: 0.40, 15: 0.39, 20: 0.38, 30: 0.38}

    _bootstrap_yield_curve(base_rate, n_instruments=30, n_iters=bootstrap_iters)
    base_npv = _price_ir_swap(trade.notional, trade.fixed_rate, trade.maturity, base_rate)

    trade_sens_list = []
    for tenor, rw in RW_GIRR.items():
        bumped_rate = base_rate + bump_size_ir * (tenor / max(trade.maturity, 0.25))
        _bootstrap_yield_curve(bumped_rate, n_instruments=30, n_iters=bootstrap_iters)
        bumped_npv = _price_ir_swap(trade.notional, trade.fixed_rate, trade.maturity, bumped_rate)
        delta_raw = bumped_npv - base_npv
        Ws = delta_raw * rw / bump_size_ir * 1e-4

        _bootstrap_yield_curve(base_rate + 0.01, n_instruments=30, n_iters=bootstrap_iters)
        _bootstrap_yield_curve(base_rate - 0.01, n_instruments=30, n_iters=bootstrap_iters)
        npv_up = _price_ir_swap(trade.notional, trade.fixed_rate, trade.maturity, base_rate + 0.01)
        npv_dn = _price_ir_swap(trade.notional, trade.fixed_rate, trade.maturity, base_rate - 0.01)
        cvr_up = npv_up - base_npv - delta_raw * 100
        cvr_dn = npv_dn - base_npv + delta_raw * 100

        trade_sens_list.append(
            Sensitivity(
                trade_id=trade.trade_id,
                risk_class="GIRR",
                bucket=trade.bucket,
                risk_factor=f"IR_{tenor}Y",
                delta=Ws,
                vega=0.0,
                curvature_up=cvr_up,
                curvature_dn=cvr_dn,
            )
        )

    _run_historical_pnl_explain(trade_sens_list, rng, n_historical_scenarios)
    return (trade.bucket, trade_sens_list)


def _compute_eq_sensitivities_for_trade(
    trade: Trade,
    base_rate: float,
    bump_size_eq: float,
    bump_size_vol: float,
    n_mc_paths: int,
    n_historical_scenarios: int,
) -> Tuple[int, List[Sensitivity]]:
    rng = np.random.default_rng(int(trade.trade_id.split("_")[1]))
    RW_EQ = {1: 0.55, 2: 0.35, 3: 0.45, 4: 0.30, 5: 0.20, 6: 0.70}
    strike = trade.spot * rng.uniform(0.9, 1.1)
    rw = RW_EQ.get(trade.bucket, 0.35)

    spot_bumps = [trade.spot * (1 + k * bump_size_eq) for k in range(-4, 6)]
    npvs = [
        _price_equity(trade.notional, s, trade.vol, trade.maturity, strike, base_rate, n_paths=n_mc_paths)
        for s in spot_bumps
    ]
    base_npv = npvs[4]
    delta_raw = npvs[5] - npvs[4]
    Ws_delta = delta_raw * rw / bump_size_eq

    vol_bumps = [trade.vol + k * bump_size_vol for k in range(-2, 3)]
    vol_npvs = [
        _price_equity(trade.notional, trade.spot, v, trade.maturity, strike, base_rate, n_paths=n_mc_paths)
        for v in vol_bumps
    ]
    vega_raw = vol_npvs[3] - vol_npvs[2]
    Ws_vega = vega_raw * 0.78 / bump_size_vol

    shock = rw * trade.spot
    npv_up = _price_equity(
        trade.notional, trade.spot + shock, trade.vol, trade.maturity, strike, base_rate, n_paths=n_mc_paths
    )
    npv_dn = _price_equity(
        trade.notional, trade.spot - shock, trade.vol, trade.maturity, strike, base_rate, n_paths=n_mc_paths
    )
    cvr_up = npv_up - base_npv - delta_raw * shock / (bump_size_eq * trade.spot)
    cvr_dn = npv_dn - base_npv + delta_raw * shock / (bump_size_eq * trade.spot)

    trade_sens = [
        Sensitivity(
            trade_id=trade.trade_id,
            risk_class="EQ",
            bucket=trade.bucket,
            risk_factor=f"EQ_B{trade.bucket}",
            delta=Ws_delta,
            vega=Ws_vega,
            curvature_up=cvr_up,
            curvature_dn=cvr_dn,
        )
    ]
    _run_historical_pnl_explain(trade_sens, rng, n_historical_scenarios)
    return (trade.bucket, trade_sens)


def _compute_csr_sensitivities_for_trade(
    trade: Trade, base_rate: float, bump_size_cs: float, n_historical_scenarios: int, bootstrap_iters: int = 5
) -> Tuple[int, List[Sensitivity]]:
    rng = np.random.default_rng(int(trade.trade_id.split("_")[1]))
    cs_tenors = [0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30]
    is_ig = trade.credit_spread < 0.01
    RW_CSR = 0.005 if is_ig else 0.05
    cs_shock = 0.06 if is_ig else 0.10

    trade_sens_list = []
    for cs_t in cs_tenors:
        if cs_t > trade.maturity + 1:
            continue

        base_npv = _price_credit_bond(trade.notional, trade.fixed_rate, trade.maturity, base_rate, trade.credit_spread)
        bump_spread = trade.credit_spread + bump_size_cs
        bumped_npv = _price_credit_bond(trade.notional, trade.fixed_rate, trade.maturity, base_rate, bump_spread)
        delta_raw = bumped_npv - base_npv
        Ws = delta_raw * RW_CSR / bump_size_cs

        npv_up = _price_credit_bond(
            trade.notional, trade.fixed_rate, trade.maturity, base_rate, trade.credit_spread + cs_shock
        )
        npv_dn = _price_credit_bond(
            trade.notional, trade.fixed_rate, trade.maturity, base_rate, max(trade.credit_spread - cs_shock, 0)
        )
        cvr_up = npv_up - base_npv - delta_raw * cs_shock / bump_size_cs
        cvr_dn = npv_dn - base_npv + delta_raw * cs_shock / bump_size_cs

        _bootstrap_yield_curve(base_rate + trade.credit_spread, n_instruments=30, n_iters=bootstrap_iters)

        trade_sens_list.append(
            Sensitivity(
                trade_id=trade.trade_id,
                risk_class="CSR",
                bucket=trade.bucket,
                risk_factor=f"CS_{cs_t}Y_{trade.currency}",
                delta=Ws,
                vega=0.0,
                curvature_up=cvr_up,
                curvature_dn=cvr_dn,
            )
        )

    _run_historical_pnl_explain(trade_sens_list, rng, n_historical_scenarios)
    return (trade.bucket, trade_sens_list)


def _compute_fx_sensitivities_for_trade(
    trade: Trade, base_rate: float, bump_size_eq: float, n_historical_scenarios: int, bootstrap_iters: int = 5
) -> Tuple[int, List[Sensitivity]]:
    rng = np.random.default_rng(int(trade.trade_id.split("_")[1]))
    vol_strikes = [0.10, 0.25, 0.50, 0.75, 0.90]
    fx_tenors = [1.0, 2.0]

    trade_sens_list = []
    for fx_t in fx_tenors:
        for vk in vol_strikes:
            base_npv = trade.notional * trade.spot
            bumped_npv = trade.notional * trade.spot * (1 + bump_size_eq)
            delta_raw = bumped_npv - base_npv
            Ws = delta_raw * 0.15 / bump_size_eq

            npv_up = trade.notional * trade.spot * 1.15
            npv_dn = trade.notional * trade.spot * 0.85
            cvr_up = npv_up - base_npv - delta_raw * 15
            cvr_dn = npv_dn - base_npv + delta_raw * 15

            _bootstrap_yield_curve(base_rate, n_instruments=15, n_iters=bootstrap_iters)

            trade_sens_list.append(
                Sensitivity(
                    trade_id=trade.trade_id,
                    risk_class="FX",
                    bucket=trade.bucket,
                    risk_factor=f"FX_{trade.currency}USD_T{fx_t}_K{vk}",
                    delta=Ws,
                    vega=0.0,
                    curvature_up=cvr_up,
                    curvature_dn=cvr_dn,
                )
            )

    _run_historical_pnl_explain(trade_sens_list, rng, n_historical_scenarios)
    return (trade.bucket, trade_sens_list)


def _compute_cmdty_sensitivities_for_trade(
    trade: Trade, base_rate: float, bump_size_eq: float, n_historical_scenarios: int, bootstrap_iters: int = 5
) -> Tuple[int, List[Sensitivity]]:
    rng = np.random.default_rng(int(trade.trade_id.split("_")[1]))
    RW_CMDTY = {1: 0.30, 2: 0.35, 3: 0.60, 4: 0.80, 5: 0.40, 6: 0.45, 7: 0.20, 8: 0.35, 9: 0.25}
    cmdty_tenors = [0.25, 0.5, 1, 2, 3, 5]
    grade_diffs = [-0.02, 0.0, 0.02]
    rw = RW_CMDTY.get(trade.bucket, 0.40)

    trade_sens_list = []
    for ct in cmdty_tenors:
        for gd in grade_diffs:
            spot_adj = trade.spot * (1 + gd)
            base_npv = trade.notional * spot_adj
            bumped_npv = trade.notional * spot_adj * (1 + bump_size_eq)
            delta_raw = bumped_npv - base_npv
            Ws = delta_raw * rw / bump_size_eq

            _bootstrap_yield_curve(base_rate + gd, n_instruments=20, n_iters=bootstrap_iters)

            trade_sens_list.append(
                Sensitivity(
                    trade_id=trade.trade_id,
                    risk_class="CMDTY",
                    bucket=trade.bucket,
                    risk_factor=f"CMDTY_B{trade.bucket}_T{ct}_G{gd:.2f}",
                    delta=Ws,
                    vega=0.0,
                    curvature_up=0.0,
                    curvature_dn=0.0,
                )
            )

    _run_historical_pnl_explain(trade_sens_list, rng, n_historical_scenarios)
    return (trade.bucket, trade_sens_list)


# ─── Unified per-trade dispatch ──────────────────────────────────────────────


def _compute_sensitivities_for_trade(
    trade: Trade,
    base_rate: float = 0.04,
    bump_size_ir: float = 0.0001,
    bump_size_eq: float = 0.01,
    bump_size_vol: float = 0.01,
    bump_size_cs: float = 0.0001,
    n_mc_paths: int = 100,
    n_historical_scenarios: int = 250,
    bootstrap_iters: int = 5,
) -> Tuple[str, int, List[Sensitivity]]:
    ac = trade.asset_class
    if ac == "GIRR":
        bid, sens = _compute_girr_sensitivities_for_trade(
            trade, base_rate, bump_size_ir, n_historical_scenarios, bootstrap_iters
        )
    elif ac == "EQ":
        bid, sens = _compute_eq_sensitivities_for_trade(
            trade, base_rate, bump_size_eq, bump_size_vol, n_mc_paths, n_historical_scenarios
        )
    elif ac == "CSR":
        bid, sens = _compute_csr_sensitivities_for_trade(
            trade, base_rate, bump_size_cs, n_historical_scenarios, bootstrap_iters
        )
    elif ac == "FX":
        bid, sens = _compute_fx_sensitivities_for_trade(
            trade, base_rate, bump_size_eq, n_historical_scenarios, bootstrap_iters
        )
    elif ac == "CMDTY":
        bid, sens = _compute_cmdty_sensitivities_for_trade(
            trade, base_rate, bump_size_eq, n_historical_scenarios, bootstrap_iters
        )
    else:
        raise ValueError(f"Unknown asset class: {ac}")
    return (ac, bid, sens)


@pf.parallel(
    split=pf.per_argument(all_trades=pf.py_list.by_chunk), combine_with=pf.py_list.concat, fixed_partition_size=1
)
def _compute_all_sensitivities_parallel(
    all_trades: List[Trade],
    base_rate: float = 0.04,
    bump_size_ir: float = 0.0001,
    bump_size_eq: float = 0.01,
    bump_size_vol: float = 0.01,
    bump_size_cs: float = 0.0001,
    n_mc_paths: int = 100,
    n_historical_scenarios: int = 250,
    bootstrap_iters: int = 5,
) -> List[Tuple[str, int, List[Sensitivity]]]:
    return [
        _compute_sensitivities_for_trade(
            t, base_rate, bump_size_ir, bump_size_eq, bump_size_vol, bump_size_cs,
            n_mc_paths, n_historical_scenarios, bootstrap_iters,
        )
        for t in all_trades
    ]


def _group_sensitivities(results: List[Tuple[str, int, List[Sensitivity]]]) -> Dict[str, BucketedSensitivities]:
    by_class: Dict[str, Dict[int, List[Sensitivity]]] = {}
    trade_counts: Dict[str, int] = {}
    for risk_class, bucket_id, sens_list in results:
        if risk_class not in by_class:
            by_class[risk_class] = {}
            trade_counts[risk_class] = 0
        trade_counts[risk_class] += 1
        if bucket_id not in by_class[risk_class]:
            by_class[risk_class][bucket_id] = []
        by_class[risk_class][bucket_id].extend(sens_list)
    result = {}
    for rc in ["GIRR", "CSR", "EQ", "FX", "CMDTY"]:
        if rc in by_class:
            result[rc] = BucketedSensitivities(risk_class=rc, by_bucket=by_class[rc], n_trades=trade_counts[rc])
        else:
            result[rc] = BucketedSensitivities(risk_class=rc, by_bucket={}, n_trades=0)
    return result


# ─── Bucket capital computation ──────────────────────────────────────────────


def _compute_single_bucket_capital(
    bucket_id: int, sens_list: List[Sensitivity], rho_eff: float
) -> Tuple[int, float, float]:
    if not sens_list:
        return (bucket_id, 0.0, 0.0)

    Ws = np.array([s.delta for s in sens_list])
    n = len(Ws)

    corr_matrix = np.full((n, n), rho_eff)
    np.fill_diagonal(corr_matrix, 1.0)
    corr_matrix += np.eye(n) * 1e-8

    try:
        L = np.linalg.cholesky(corr_matrix)
        Lw = np.linalg.solve(L, Ws)
        Kb_sq = float(Lw @ Lw)
    except np.linalg.LinAlgError:
        Kb_sq = float(Ws @ corr_matrix @ Ws)

    cvr_up = np.array([s.curvature_up for s in sens_list])
    cvr_dn = np.array([s.curvature_dn for s in sens_list])
    curvature_capital = max(0.0, -float(np.minimum(cvr_up, cvr_dn).sum()))

    Kb = float(np.sqrt(max(Kb_sq, 0.0))) + curvature_capital
    Sb = float(Ws.sum())
    return (bucket_id, Kb, Sb)


def compute_bucket_capital_under_scenario(
    bucketed: BucketedSensitivities, corr_scenario: str
) -> BucketCapital:
    scenario_scalar = CORR_SCENARIOS[corr_scenario]
    risk_class = bucketed.risk_class

    INTRA_CORR = {"GIRR": 0.999, "CSR": 0.35, "EQ": 0.15, "FX": 0.60, "CMDTY": 0.20}
    rho_base = INTRA_CORR.get(risk_class, 0.30)
    rho_eff = min(max(rho_base * scenario_scalar, 0.0), 1.0)

    CROSS_BUCKET_CORR = {"GIRR": 0.0, "CSR": 0.0, "EQ": 0.15, "FX": 0.60, "CMDTY": 0.20}
    gamma = min(max(CROSS_BUCKET_CORR.get(risk_class, 0.0) * scenario_scalar, 0.0), 1.0)

    bucket_items = list(bucketed.by_bucket.items())
    results = [_compute_single_bucket_capital(bid, slist, rho_eff) for bid, slist in bucket_items]

    kb_per_bucket: Dict[int, float] = {bid: Kb for bid, Kb, _ in results}
    Sb_per_bucket: Dict[int, float] = {bid: Sb for bid, _, Sb in results}

    buckets = sorted(kb_per_bucket.keys())
    Kb_vec = np.array([kb_per_bucket[b] for b in buckets])
    Sb_vec = np.array([Sb_per_bucket[b] for b in buckets])

    n_b = len(buckets)
    if n_b == 0:
        Ks = 0.0
    elif n_b == 1:
        Ks = float(Kb_vec[0])
    else:
        cross_corr = np.full((n_b, n_b), gamma)
        np.fill_diagonal(cross_corr, 1.0)
        Ks_sq = float(Kb_vec @ Kb_vec) + float(Sb_vec @ cross_corr @ Sb_vec) - float(Sb_vec @ Sb_vec)
        Ks = float(np.sqrt(max(Ks_sq, 0.0)))

    return BucketCapital(risk_class=risk_class, corr_scenario=corr_scenario, kb_per_bucket=kb_per_bucket, ks=Ks)


def worst_case_risk_class_capital(low: BucketCapital, medium: BucketCapital, high: BucketCapital) -> RiskClassCapital:
    scenario_ks = {bc.corr_scenario: bc.ks for bc in [low, medium, high]}
    worst_scenario = max(scenario_ks, key=scenario_ks.get)
    return RiskClassCapital(
        risk_class=low.risk_class,
        capital=scenario_ks[worst_scenario],
        worst_scenario=worst_scenario,
        scenario_capitals=scenario_ks,
    )


def aggregate_total_capital(
    risk_class_capitals_girr: RiskClassCapital,
    risk_class_capitals_csr: RiskClassCapital,
    risk_class_capitals_eq: RiskClassCapital,
    risk_class_capitals_fx: RiskClassCapital,
    risk_class_capitals_cmdty: RiskClassCapital,
    trades: List[Trade],
) -> FRTBCapitalReport:
    ORDER = ["GIRR", "CSR", "EQ", "FX", "CMDTY"]
    rc_cap = {
        rc.risk_class: rc.capital
        for rc in [
            risk_class_capitals_girr,
            risk_class_capitals_csr,
            risk_class_capitals_eq,
            risk_class_capitals_fx,
            risk_class_capitals_cmdty,
        ]
    }

    Ks_vec = np.array([rc_cap.get(rc, 0.0) for rc in ORDER])
    total_sq = float(Ks_vec @ CROSS_CLASS_CORR @ Ks_vec)
    total_capital = float(np.sqrt(max(total_sq, 0.0)))

    by_risk_class = {rc: float(Ks_vec[i]) for i, rc in enumerate(ORDER)}

    rows = []
    for rc in [
        risk_class_capitals_girr,
        risk_class_capitals_csr,
        risk_class_capitals_eq,
        risk_class_capitals_fx,
        risk_class_capitals_cmdty,
    ]:
        for scenario, cap in rc.scenario_capitals.items():
            rows.append(
                {
                    "Risk Class": rc.risk_class,
                    "Scenario": scenario,
                    "Capital ($M)": cap / 1e6,
                    "Is Worst": scenario == rc.worst_scenario,
                }
            )
    scenario_df = pd.DataFrame(rows)

    asset_classes = [t.asset_class for t in trades]
    sens_df = pd.DataFrame({"Asset Class": asset_classes}).value_counts().reset_index()
    sens_df.columns = ["Asset Class", "Trade Count"]

    return FRTBCapitalReport(
        total_capital=total_capital,
        by_risk_class=by_risk_class,
        scenario_breakdown=scenario_df,
        sensitivity_summary=sens_df,
    )


# ─── ParFun pipeline ────────────────────────────────────────────────────────


def frtb_sbm_capital_parfun(
    n_trades: int = N_TRADES, seed: int = 42, bootstrap_iters: int = BOOTSTRAP_ITERS
) -> FRTBCapitalReport:
    trades = load_and_enrich_trades(n_trades=n_trades, seed=seed)

    cluster = SchedulerClusterCombo(n_workers=16, logging_level="WARNING")
    try:
        scheduler_address = cluster.get_address()

        # Flat parfun: ALL trades in one batch → every worker busy
        with pf.set_parallel_backend_context("scaler_remote", scheduler_address=scheduler_address):
            all_results = _compute_all_sensitivities_parallel(
                trades, n_mc_paths=EQ_MC_PATHS, bootstrap_iters=bootstrap_iters
            )
    finally:
        cluster.shutdown()

    grouped = _group_sensitivities(all_results)

    # Post-sensitivity pipeline runs locally (sub-second)
    RISK_CLASSES = ["GIRR", "CSR", "EQ", "FX", "CMDTY"]
    rc_capitals = {}
    for rc in RISK_CLASSES:
        bucketed = grouped[rc]
        caps = {}
        for scenario in ["low", "medium", "high"]:
            caps[scenario] = compute_bucket_capital_under_scenario(bucketed, scenario)
        rc_capitals[rc] = worst_case_risk_class_capital(caps["low"], caps["medium"], caps["high"])

    return aggregate_total_capital(
        rc_capitals["GIRR"], rc_capitals["CSR"], rc_capitals["EQ"], rc_capitals["FX"], rc_capitals["CMDTY"], trades
    )


t0 = time.perf_counter()
report = frtb_sbm_capital_parfun()
elapsed = time.perf_counter() - t0

print(f"Total FRTB SBM Capital: ${report.total_capital / 1e6:,.4f}M")
print(f"By risk class: { {k: f'${v/1e6:,.4f}M' for k, v in report.by_risk_class.items()} }")
print(f"Wall time: {elapsed:.1f}s")


# Diff: Native vs ParFun

In [ ]:
import difflib
import json
import re

from IPython.display import HTML, display

with open("FRTB.ipynb") as _f:
    _nb = json.load(_f)
native_code = "".join(_nb["cells"][3]["source"])
parallel_code = "".join(_nb["cells"][5]["source"])

differ = difflib.HtmlDiff(wrapcolumn=90)
table = differ.make_table(
    native_code.splitlines(),
    parallel_code.splitlines(),
    fromdesc="Native Python",
    todesc="With ParFun",
    context=False,
)

# Strip unwanted columns: navigation links, line numbers, and nowrap attribute
table = re.sub(r'<td class="diff_next"[^>]*>.*?</td>', "", table)
table = re.sub(r'<th class="diff_next"[^>]*>.*?</th>', "", table)
table = re.sub(r'<td class="diff_header"[^>]*>.*?</td>', "", table)
table = table.replace('colspan="2"', "")
table = table.replace(' nowrap="nowrap"', "")
table = re.sub(
    r"(\s*<colgroup></colgroup>){6}",
    "\n        <colgroup></colgroup> <colgroup></colgroup>",
    table,
)

css = """<style>
    table.diff { font-family: monospace; font-size: 10px; border-collapse: collapse; width: 100%; }
    table.diff td { padding: 2px 8px; white-space: pre-wrap; word-break: break-all; text-align: left !important; }
    table.diff th { padding: 6px 8px; text-align: center; background-color: #f0f0f0; font-size: 14px; }
    td.diff_add { background-color: #e6ffec; }
    td.diff_chg { background-color: #fff8c5; }
    td.diff_sub { background-color: #ffebe9; }
    span.diff_add { background-color: #ccffd8; }
    span.diff_chg { background-color: #fff2a8; }
    span.diff_sub { background-color: #ffc0c0; }
</style>"""

display(HTML(css + table))

Native Python,With ParFun
import os,import os
,
"os.environ[""OPENBLAS_NUM_THREADS""] = ""1""","os.environ[""OPENBLAS_NUM_THREADS""] = ""1"""
"os.environ[""GOTO_NUM_THREADS""] = ""1""","os.environ[""GOTO_NUM_THREADS""] = ""1"""
"os.environ[""OMP_NUM_THREADS""] = ""1""","os.environ[""OMP_NUM_THREADS""] = ""1"""
"os.environ[""MKL_NUM_THREADS""] = ""1""","os.environ[""MKL_NUM_THREADS""] = ""1"""
,
import math,import math
import random,import random
import struct,import struct


# ParGraph Implementation

In [ ]:
import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["GOTO_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import math
import random
import struct
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from pargraph import GraphEngine
from scaler import Client, SchedulerClusterCombo


# ─── Tunable parameters ─────────────────────────────────────────────────────

N_TRADES = 100
EQ_MC_PATHS = 100
BOOTSTRAP_ITERS = 1000

RISK_CLASSES = ["GIRR", "CSR", "EQ", "FX", "CMDTY"]


# ─── Regulatory parameters (Basel IV FRTB CRE52) ────────────────────────────

GIRR_TENORS = [0.25, 0.5, 1, 2, 3, 5, 10, 15, 20, 30]

EQ_BUCKETS = {
    1: "Large-cap EM",
    2: "Large-cap DM",
    3: "Small-cap EM",
    4: "Small-cap DM",
    5: "Indices ETF",
    6: "Volatility",
}

CROSS_CLASS_CORR = np.array(
    [
        [1.00, 0.01, 0.04, 0.04, 0.04],
        [0.01, 1.00, 0.05, 0.04, 0.08],
        [0.04, 0.05, 1.00, 0.15, 0.20],
        [0.04, 0.04, 0.15, 1.00, 0.08],
        [0.04, 0.08, 0.20, 0.08, 1.00],
    ]
)

CORR_SCENARIOS = {
    "low": 0.75,
    "medium": 1.00,
    "high": 1.25,
}


# ─── Domain types ────────────────────────────────────────────────────────────


@dataclass
class Trade:
    trade_id: str
    asset_class: str
    instrument: str
    notional: float
    currency: str
    maturity: float
    fixed_rate: float
    ticker: str
    bucket: int
    spot: float
    vol: float
    credit_spread: float


@dataclass
class Sensitivity:
    trade_id: str
    risk_class: str
    bucket: int
    risk_factor: str
    delta: float
    vega: float
    curvature_up: float
    curvature_dn: float


@dataclass
class BucketedSensitivities:
    risk_class: str
    by_bucket: Dict[int, List[Sensitivity]]
    n_trades: int


@dataclass
class BucketCapital:
    risk_class: str
    corr_scenario: str
    kb_per_bucket: Dict[int, float]
    ks: float


@dataclass
class RiskClassCapital:
    risk_class: str
    capital: float
    worst_scenario: str
    scenario_capitals: Dict[str, float]


@dataclass
class FRTBCapitalReport:
    total_capital: float
    by_risk_class: Dict[str, float]
    scenario_breakdown: pd.DataFrame
    sensitivity_summary: pd.DataFrame


# ─── Pricing functions ───────────────────────────────────────────────────────


def _price_ir_swap(notional: float, fixed_rate: float, maturity: float, flat_rate: float) -> float:
    T = maturity
    annuity = sum(np.exp(-flat_rate * t) for t in np.arange(1, T + 1))
    par_rate = (1 - np.exp(-flat_rate * T)) / annuity
    return notional * (par_rate - fixed_rate) * annuity


def _price_equity(
    notional: float,
    spot: float,
    vol: float,
    maturity: float,
    strike: float,
    rate: float,
    flag: str = "call",
    n_paths: int = 5_000,
) -> float:
    seed_bytes = struct.pack("ddddd", spot, vol, maturity, strike, rate)
    local_rng = random.Random(seed_bytes)

    if maturity <= 0 or vol <= 0:
        return 0.0
    dt = maturity / 252.0
    n_steps = max(int(maturity * 252), 1)

    payoff_sum = 0.0
    drift = (rate - 0.5 * vol**2) * dt
    v_sqrt_dt = vol * math.sqrt(dt)

    for i in range(n_paths):
        path_ret = 0.0
        for j in range(n_steps):
            z = local_rng.gauss(0.0, 1.0)
            path_ret += drift + v_sqrt_dt * z

        ST = spot * math.exp(path_ret)
        if flag == "call":
            payoff = max(ST - strike, 0.0)
        else:
            payoff = max(strike - ST, 0.0)
        payoff_sum += payoff

    return notional * math.exp(-rate * maturity) * (payoff_sum / n_paths)


def _price_credit_bond(notional: float, coupon: float, maturity: float, risk_free: float, spread: float) -> float:
    ytm = risk_free + spread
    T = int(max(maturity, 1))
    pv_coupons = sum(notional * coupon * np.exp(-ytm * t) for t in range(1, T + 1))
    pv_principal = notional * np.exp(-ytm * maturity)
    return pv_coupons + pv_principal


def _bootstrap_yield_curve(base_rate: float, n_instruments: int = 30, n_iters: int = 5) -> np.ndarray:
    seed_bytes = struct.pack("di", base_rate, n_instruments)
    local_rng = random.Random(seed_bytes)
    A = np.zeros((n_instruments, n_instruments))
    for i in range(n_instruments):
        for j in range(n_instruments):
            val = 0.0
            for k in range(n_iters):
                val += local_rng.gauss(0.0, 1.0)
            A[i, j] = val / max(n_iters, 1)

    A = A @ A.T + np.eye(n_instruments) * base_rate

    b = np.empty(n_instruments)
    for i in range(n_instruments):
        val = 0.0
        for k in range(n_iters):
            val += local_rng.gauss(0.0, 1.0)
        b[i] = val / max(n_iters, 1)

    L = np.linalg.cholesky(A)
    y = np.linalg.solve(L, b)
    discount_factors = np.linalg.solve(L.T, y)
    return discount_factors


def _run_historical_pnl_explain(sensitivities_list, rng, n_scenarios: int = 250) -> float:
    if not sensitivities_list:
        return 0.0
    n_sens = len(sensitivities_list)
    rf_moves = rng.standard_normal((n_scenarios, n_sens)) * 0.001
    ws_vec = np.array([s.delta for s in sensitivities_list])
    rtpl = rf_moves @ ws_vec
    hpnl = rtpl + rng.standard_normal(n_scenarios) * abs(ws_vec).mean() * 0.05
    residuals = hpnl - rtpl
    resid_matrix = np.outer(residuals, residuals) / n_scenarios
    _ = np.linalg.svd(resid_matrix)
    return float(np.sqrt(np.mean(residuals**2)))


# ─── Trade generation ────────────────────────────────────────────────────────


def load_and_enrich_trades(n_trades: int, seed: int = 42) -> List[Trade]:
    rng = np.random.default_rng(seed)
    asset_classes = ["GIRR", "CSR", "EQ", "FX", "CMDTY"]
    weights = [0.35, 0.25, 0.20, 0.12, 0.08]
    currencies = ["USD", "EUR", "GBP", "JPY", "CHF"]
    tickers = [f"TICKER_{i:04d}" for i in range(100)]

    trades = []
    for i in range(n_trades):
        ac = rng.choice(asset_classes, p=weights)
        trades.append(
            Trade(
                trade_id=f"TRD_{i:06d}",
                asset_class=ac,
                instrument={
                    "GIRR": "ir_swap",
                    "CSR": "bond",
                    "EQ": "equity_option",
                    "FX": "fx_option",
                    "CMDTY": "commodity_fwd",
                }[ac],
                notional=float(rng.choice([1e5, 5e5, 1e6, 5e6, 10e6])),
                currency=rng.choice(currencies),
                maturity=float(rng.uniform(0.5, 30.0)),
                fixed_rate=float(rng.uniform(0.02, 0.07)),
                ticker=rng.choice(tickers),
                bucket=int(rng.integers(1, 7)),
                spot=float(rng.uniform(50, 200)),
                vol=float(rng.uniform(0.10, 0.60)),
                credit_spread=float(rng.uniform(0.005, 0.025)),
            )
        )
    return trades


# ─── Per-trade sensitivity functions ─────────────────────────────────────────


def _compute_girr_sensitivities_for_trade(
    trade: Trade, base_rate: float, bump_size_ir: float, n_historical_scenarios: int, bootstrap_iters: int = 5
) -> Tuple[int, List[Sensitivity]]:
    rng = np.random.default_rng(int(trade.trade_id.split("_")[1]))
    RW_GIRR = {0.25: 1.74, 0.5: 1.74, 1: 0.74, 2: 0.58, 3: 0.49, 5: 0.44, 10: 0.40, 15: 0.39, 20: 0.38, 30: 0.38}

    _bootstrap_yield_curve(base_rate, n_instruments=30, n_iters=bootstrap_iters)
    base_npv = _price_ir_swap(trade.notional, trade.fixed_rate, trade.maturity, base_rate)

    trade_sens_list = []
    for tenor, rw in RW_GIRR.items():
        bumped_rate = base_rate + bump_size_ir * (tenor / max(trade.maturity, 0.25))
        _bootstrap_yield_curve(bumped_rate, n_instruments=30, n_iters=bootstrap_iters)
        bumped_npv = _price_ir_swap(trade.notional, trade.fixed_rate, trade.maturity, bumped_rate)
        delta_raw = bumped_npv - base_npv
        Ws = delta_raw * rw / bump_size_ir * 1e-4

        _bootstrap_yield_curve(base_rate + 0.01, n_instruments=30, n_iters=bootstrap_iters)
        _bootstrap_yield_curve(base_rate - 0.01, n_instruments=30, n_iters=bootstrap_iters)
        npv_up = _price_ir_swap(trade.notional, trade.fixed_rate, trade.maturity, base_rate + 0.01)
        npv_dn = _price_ir_swap(trade.notional, trade.fixed_rate, trade.maturity, base_rate - 0.01)
        cvr_up = npv_up - base_npv - delta_raw * 100
        cvr_dn = npv_dn - base_npv + delta_raw * 100

        trade_sens_list.append(
            Sensitivity(
                trade_id=trade.trade_id,
                risk_class="GIRR",
                bucket=trade.bucket,
                risk_factor=f"IR_{tenor}Y",
                delta=Ws,
                vega=0.0,
                curvature_up=cvr_up,
                curvature_dn=cvr_dn,
            )
        )

    _run_historical_pnl_explain(trade_sens_list, rng, n_historical_scenarios)
    return (trade.bucket, trade_sens_list)


def _compute_eq_sensitivities_for_trade(
    trade: Trade,
    base_rate: float,
    bump_size_eq: float,
    bump_size_vol: float,
    n_mc_paths: int,
    n_historical_scenarios: int,
) -> Tuple[int, List[Sensitivity]]:
    rng = np.random.default_rng(int(trade.trade_id.split("_")[1]))
    RW_EQ = {1: 0.55, 2: 0.35, 3: 0.45, 4: 0.30, 5: 0.20, 6: 0.70}
    strike = trade.spot * rng.uniform(0.9, 1.1)
    rw = RW_EQ.get(trade.bucket, 0.35)

    spot_bumps = [trade.spot * (1 + k * bump_size_eq) for k in range(-4, 6)]
    npvs = [
        _price_equity(trade.notional, s, trade.vol, trade.maturity, strike, base_rate, n_paths=n_mc_paths)
        for s in spot_bumps
    ]
    base_npv = npvs[4]
    delta_raw = npvs[5] - npvs[4]
    Ws_delta = delta_raw * rw / bump_size_eq

    vol_bumps = [trade.vol + k * bump_size_vol for k in range(-2, 3)]
    vol_npvs = [
        _price_equity(trade.notional, trade.spot, v, trade.maturity, strike, base_rate, n_paths=n_mc_paths)
        for v in vol_bumps
    ]
    vega_raw = vol_npvs[3] - vol_npvs[2]
    Ws_vega = vega_raw * 0.78 / bump_size_vol

    shock = rw * trade.spot
    npv_up = _price_equity(
        trade.notional, trade.spot + shock, trade.vol, trade.maturity, strike, base_rate, n_paths=n_mc_paths
    )
    npv_dn = _price_equity(
        trade.notional, trade.spot - shock, trade.vol, trade.maturity, strike, base_rate, n_paths=n_mc_paths
    )
    cvr_up = npv_up - base_npv - delta_raw * shock / (bump_size_eq * trade.spot)
    cvr_dn = npv_dn - base_npv + delta_raw * shock / (bump_size_eq * trade.spot)

    trade_sens = [
        Sensitivity(
            trade_id=trade.trade_id,
            risk_class="EQ",
            bucket=trade.bucket,
            risk_factor=f"EQ_B{trade.bucket}",
            delta=Ws_delta,
            vega=Ws_vega,
            curvature_up=cvr_up,
            curvature_dn=cvr_dn,
        )
    ]
    _run_historical_pnl_explain(trade_sens, rng, n_historical_scenarios)
    return (trade.bucket, trade_sens)


def _compute_csr_sensitivities_for_trade(
    trade: Trade, base_rate: float, bump_size_cs: float, n_historical_scenarios: int, bootstrap_iters: int = 5
) -> Tuple[int, List[Sensitivity]]:
    rng = np.random.default_rng(int(trade.trade_id.split("_")[1]))
    cs_tenors = [0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30]
    is_ig = trade.credit_spread < 0.01
    RW_CSR = 0.005 if is_ig else 0.05
    cs_shock = 0.06 if is_ig else 0.10

    trade_sens_list = []
    for cs_t in cs_tenors:
        if cs_t > trade.maturity + 1:
            continue

        base_npv = _price_credit_bond(trade.notional, trade.fixed_rate, trade.maturity, base_rate, trade.credit_spread)
        bump_spread = trade.credit_spread + bump_size_cs
        bumped_npv = _price_credit_bond(trade.notional, trade.fixed_rate, trade.maturity, base_rate, bump_spread)
        delta_raw = bumped_npv - base_npv
        Ws = delta_raw * RW_CSR / bump_size_cs

        npv_up = _price_credit_bond(
            trade.notional, trade.fixed_rate, trade.maturity, base_rate, trade.credit_spread + cs_shock
        )
        npv_dn = _price_credit_bond(
            trade.notional, trade.fixed_rate, trade.maturity, base_rate, max(trade.credit_spread - cs_shock, 0)
        )
        cvr_up = npv_up - base_npv - delta_raw * cs_shock / bump_size_cs
        cvr_dn = npv_dn - base_npv + delta_raw * cs_shock / bump_size_cs

        _bootstrap_yield_curve(base_rate + trade.credit_spread, n_instruments=30, n_iters=bootstrap_iters)

        trade_sens_list.append(
            Sensitivity(
                trade_id=trade.trade_id,
                risk_class="CSR",
                bucket=trade.bucket,
                risk_factor=f"CS_{cs_t}Y_{trade.currency}",
                delta=Ws,
                vega=0.0,
                curvature_up=cvr_up,
                curvature_dn=cvr_dn,
            )
        )

    _run_historical_pnl_explain(trade_sens_list, rng, n_historical_scenarios)
    return (trade.bucket, trade_sens_list)


def _compute_fx_sensitivities_for_trade(
    trade: Trade, base_rate: float, bump_size_eq: float, n_historical_scenarios: int, bootstrap_iters: int = 5
) -> Tuple[int, List[Sensitivity]]:
    rng = np.random.default_rng(int(trade.trade_id.split("_")[1]))
    vol_strikes = [0.10, 0.25, 0.50, 0.75, 0.90]
    fx_tenors = [1.0, 2.0]

    trade_sens_list = []
    for fx_t in fx_tenors:
        for vk in vol_strikes:
            base_npv = trade.notional * trade.spot
            bumped_npv = trade.notional * trade.spot * (1 + bump_size_eq)
            delta_raw = bumped_npv - base_npv
            Ws = delta_raw * 0.15 / bump_size_eq

            npv_up = trade.notional * trade.spot * 1.15
            npv_dn = trade.notional * trade.spot * 0.85
            cvr_up = npv_up - base_npv - delta_raw * 15
            cvr_dn = npv_dn - base_npv + delta_raw * 15

            _bootstrap_yield_curve(base_rate, n_instruments=15, n_iters=bootstrap_iters)

            trade_sens_list.append(
                Sensitivity(
                    trade_id=trade.trade_id,
                    risk_class="FX",
                    bucket=trade.bucket,
                    risk_factor=f"FX_{trade.currency}USD_T{fx_t}_K{vk}",
                    delta=Ws,
                    vega=0.0,
                    curvature_up=cvr_up,
                    curvature_dn=cvr_dn,
                )
            )

    _run_historical_pnl_explain(trade_sens_list, rng, n_historical_scenarios)
    return (trade.bucket, trade_sens_list)


def _compute_cmdty_sensitivities_for_trade(
    trade: Trade, base_rate: float, bump_size_eq: float, n_historical_scenarios: int, bootstrap_iters: int = 5
) -> Tuple[int, List[Sensitivity]]:
    rng = np.random.default_rng(int(trade.trade_id.split("_")[1]))
    RW_CMDTY = {1: 0.30, 2: 0.35, 3: 0.60, 4: 0.80, 5: 0.40, 6: 0.45, 7: 0.20, 8: 0.35, 9: 0.25}
    cmdty_tenors = [0.25, 0.5, 1, 2, 3, 5]
    grade_diffs = [-0.02, 0.0, 0.02]
    rw = RW_CMDTY.get(trade.bucket, 0.40)

    trade_sens_list = []
    for ct in cmdty_tenors:
        for gd in grade_diffs:
            spot_adj = trade.spot * (1 + gd)
            base_npv = trade.notional * spot_adj
            bumped_npv = trade.notional * spot_adj * (1 + bump_size_eq)
            delta_raw = bumped_npv - base_npv
            Ws = delta_raw * rw / bump_size_eq

            _bootstrap_yield_curve(base_rate + gd, n_instruments=20, n_iters=bootstrap_iters)

            trade_sens_list.append(
                Sensitivity(
                    trade_id=trade.trade_id,
                    risk_class="CMDTY",
                    bucket=trade.bucket,
                    risk_factor=f"CMDTY_B{trade.bucket}_T{ct}_G{gd:.2f}",
                    delta=Ws,
                    vega=0.0,
                    curvature_up=0.0,
                    curvature_dn=0.0,
                )
            )

    _run_historical_pnl_explain(trade_sens_list, rng, n_historical_scenarios)
    return (trade.bucket, trade_sens_list)


# ─── Unified per-trade dispatch ──────────────────────────────────────────────


def _compute_sensitivities_for_trade(
    trade: Trade,
    base_rate: float = 0.04,
    bump_size_ir: float = 0.0001,
    bump_size_eq: float = 0.01,
    bump_size_vol: float = 0.01,
    bump_size_cs: float = 0.0001,
    n_mc_paths: int = 100,
    n_historical_scenarios: int = 250,
    bootstrap_iters: int = 5,
) -> Tuple[str, int, List[Sensitivity]]:
    ac = trade.asset_class
    if ac == "GIRR":
        bid, sens = _compute_girr_sensitivities_for_trade(
            trade, base_rate, bump_size_ir, n_historical_scenarios, bootstrap_iters
        )
    elif ac == "EQ":
        bid, sens = _compute_eq_sensitivities_for_trade(
            trade, base_rate, bump_size_eq, bump_size_vol, n_mc_paths, n_historical_scenarios
        )
    elif ac == "CSR":
        bid, sens = _compute_csr_sensitivities_for_trade(
            trade, base_rate, bump_size_cs, n_historical_scenarios, bootstrap_iters
        )
    elif ac == "FX":
        bid, sens = _compute_fx_sensitivities_for_trade(
            trade, base_rate, bump_size_eq, n_historical_scenarios, bootstrap_iters
        )
    elif ac == "CMDTY":
        bid, sens = _compute_cmdty_sensitivities_for_trade(
            trade, base_rate, bump_size_eq, n_historical_scenarios, bootstrap_iters
        )
    else:
        raise ValueError(f"Unknown asset class: {ac}")
    return (ac, bid, sens)


# ─── ParGraph helpers ────────────────────────────────────────────────────────


def _collect_and_bucket_results(risk_class: str, *results: Tuple[str, int, List[Sensitivity]]) -> BucketedSensitivities:
    sensitivities: Dict[int, List[Sensitivity]] = {}
    for _, bucket_id, sens_list in results:
        if bucket_id not in sensitivities:
            sensitivities[bucket_id] = []
        sensitivities[bucket_id].extend(sens_list)
    return BucketedSensitivities(risk_class=risk_class, by_bucket=sensitivities, n_trades=len(results))


# ─── Bucket capital computation ──────────────────────────────────────────────


def _compute_single_bucket_capital(
    bucket_id: int, sens_list: List[Sensitivity], rho_eff: float
) -> Tuple[int, float, float]:
    if not sens_list:
        return (bucket_id, 0.0, 0.0)

    Ws = np.array([s.delta for s in sens_list])
    n = len(Ws)

    corr_matrix = np.full((n, n), rho_eff)
    np.fill_diagonal(corr_matrix, 1.0)
    corr_matrix += np.eye(n) * 1e-8

    try:
        L = np.linalg.cholesky(corr_matrix)
        Lw = np.linalg.solve(L, Ws)
        Kb_sq = float(Lw @ Lw)
    except np.linalg.LinAlgError:
        Kb_sq = float(Ws @ corr_matrix @ Ws)

    cvr_up = np.array([s.curvature_up for s in sens_list])
    cvr_dn = np.array([s.curvature_dn for s in sens_list])
    curvature_capital = max(0.0, -float(np.minimum(cvr_up, cvr_dn).sum()))

    Kb = float(np.sqrt(max(Kb_sq, 0.0))) + curvature_capital
    Sb = float(Ws.sum())
    return (bucket_id, Kb, Sb)


def compute_bucket_capital_under_scenario(
    bucketed: BucketedSensitivities, corr_scenario: str, scheduler_address=None
) -> BucketCapital:
    scenario_scalar = CORR_SCENARIOS[corr_scenario]
    risk_class = bucketed.risk_class

    INTRA_CORR = {"GIRR": 0.999, "CSR": 0.35, "EQ": 0.15, "FX": 0.60, "CMDTY": 0.20}
    rho_base = INTRA_CORR.get(risk_class, 0.30)
    rho_eff = min(max(rho_base * scenario_scalar, 0.0), 1.0)

    CROSS_BUCKET_CORR = {"GIRR": 0.0, "CSR": 0.0, "EQ": 0.15, "FX": 0.60, "CMDTY": 0.20}
    gamma = min(max(CROSS_BUCKET_CORR.get(risk_class, 0.0) * scenario_scalar, 0.0), 1.0)

    bucket_items = list(bucketed.by_bucket.items())
    results = [_compute_single_bucket_capital(bid, slist, rho_eff) for bid, slist in bucket_items]

    kb_per_bucket: Dict[int, float] = {bid: Kb for bid, Kb, _ in results}
    Sb_per_bucket: Dict[int, float] = {bid: Sb for bid, _, Sb in results}

    buckets = sorted(kb_per_bucket.keys())
    Kb_vec = np.array([kb_per_bucket[b] for b in buckets])
    Sb_vec = np.array([Sb_per_bucket[b] for b in buckets])

    n_b = len(buckets)
    if n_b == 0:
        Ks = 0.0
    elif n_b == 1:
        Ks = float(Kb_vec[0])
    else:
        cross_corr = np.full((n_b, n_b), gamma)
        np.fill_diagonal(cross_corr, 1.0)
        Ks_sq = float(Kb_vec @ Kb_vec) + float(Sb_vec @ cross_corr @ Sb_vec) - float(Sb_vec @ Sb_vec)
        Ks = float(np.sqrt(max(Ks_sq, 0.0)))

    return BucketCapital(risk_class=risk_class, corr_scenario=corr_scenario, kb_per_bucket=kb_per_bucket, ks=Ks)


def worst_case_risk_class_capital(low: BucketCapital, medium: BucketCapital, high: BucketCapital) -> RiskClassCapital:
    scenario_ks = {bc.corr_scenario: bc.ks for bc in [low, medium, high]}
    worst_scenario = max(scenario_ks, key=scenario_ks.get)
    return RiskClassCapital(
        risk_class=low.risk_class,
        capital=scenario_ks[worst_scenario],
        worst_scenario=worst_scenario,
        scenario_capitals=scenario_ks,
    )


def aggregate_total_capital(
    risk_class_capitals_girr: RiskClassCapital,
    risk_class_capitals_csr: RiskClassCapital,
    risk_class_capitals_eq: RiskClassCapital,
    risk_class_capitals_fx: RiskClassCapital,
    risk_class_capitals_cmdty: RiskClassCapital,
    trades: List[Trade],
) -> FRTBCapitalReport:
    ORDER = ["GIRR", "CSR", "EQ", "FX", "CMDTY"]
    rc_cap = {
        rc.risk_class: rc.capital
        for rc in [
            risk_class_capitals_girr,
            risk_class_capitals_csr,
            risk_class_capitals_eq,
            risk_class_capitals_fx,
            risk_class_capitals_cmdty,
        ]
    }

    Ks_vec = np.array([rc_cap.get(rc, 0.0) for rc in ORDER])
    total_sq = float(Ks_vec @ CROSS_CLASS_CORR @ Ks_vec)
    total_capital = float(np.sqrt(max(total_sq, 0.0)))

    by_risk_class = {rc: float(Ks_vec[i]) for i, rc in enumerate(ORDER)}

    rows = []
    for rc in [
        risk_class_capitals_girr,
        risk_class_capitals_csr,
        risk_class_capitals_eq,
        risk_class_capitals_fx,
        risk_class_capitals_cmdty,
    ]:
        for scenario, cap in rc.scenario_capitals.items():
            rows.append(
                {
                    "Risk Class": rc.risk_class,
                    "Scenario": scenario,
                    "Capital ($M)": cap / 1e6,
                    "Is Worst": scenario == rc.worst_scenario,
                }
            )
    scenario_df = pd.DataFrame(rows)

    asset_classes = [t.asset_class for t in trades]
    sens_df = pd.DataFrame({"Asset Class": asset_classes}).value_counts().reset_index()
    sens_df.columns = ["Asset Class", "Trade Count"]

    return FRTBCapitalReport(
        total_capital=total_capital,
        by_risk_class=by_risk_class,
        scenario_breakdown=scenario_df,
        sensitivity_summary=sens_df,
    )


# ─── ParGraph DAG builder ───────────────────────────────────────────────────


def _build_per_trade_graph(trades: List[Trade], bootstrap_iters: int) -> dict:
    dg: dict = {}

    # Per-trade sensitivity nodes (each scheduled independently by pargraph)
    for i, trade in enumerate(trades):
        dg[f"sens_{i}"] = (
            _compute_sensitivities_for_trade,
            trade,
            0.04, 0.0001, 0.01, 0.01, 0.0001,
            EQ_MC_PATHS, 250, bootstrap_iters,
        )

    # Per-risk-class collector nodes (each depends only on its own trades)
    for rc in RISK_CLASSES:
        rc_indices = [i for i, t in enumerate(trades) if t.asset_class == rc]
        dg[f"bucketed_{rc}"] = (_collect_and_bucket_results, rc, *[f"sens_{i}" for i in rc_indices])

        # 3 correlation scenario nodes per risk class (15 total)
        for scenario in ["low", "medium", "high"]:
            dg[f"capital_{rc}_{scenario}"] = (
                compute_bucket_capital_under_scenario,
                f"bucketed_{rc}",
                scenario,
                None,
            )

        # Worst-case selection per risk class
        dg[f"worst_{rc}"] = (
            worst_case_risk_class_capital,
            f"capital_{rc}_low",
            f"capital_{rc}_medium",
            f"capital_{rc}_high",
        )

    # Final cross-class aggregation
    dg["trades_data"] = trades
    dg["total"] = (
        aggregate_total_capital,
        "worst_GIRR",
        "worst_CSR",
        "worst_EQ",
        "worst_FX",
        "worst_CMDTY",
        "trades_data",
    )
    return dg


# ─── ParGraph pipeline ──────────────────────────────────────────────────────


def frtb_sbm_capital_pargraph(
    n_trades: int = N_TRADES, seed: int = 42, bootstrap_iters: int = BOOTSTRAP_ITERS
) -> FRTBCapitalReport:
    trades = load_and_enrich_trades(n_trades=n_trades, seed=seed)
    dict_graph = _build_per_trade_graph(trades, bootstrap_iters)

    cluster = SchedulerClusterCombo(n_workers=16, logging_level="WARNING")
    try:
        client = Client(address=cluster.get_address())
        engine = GraphEngine(backend=client)
        (report,) = engine.get(dict_graph, ["total"])
        client.disconnect()
    finally:
        cluster.shutdown()
    return report


t0 = time.perf_counter()
report = frtb_sbm_capital_pargraph()
elapsed = time.perf_counter() - t0

print(f"Total FRTB SBM Capital: ${report.total_capital / 1e6:,.4f}M")
print(f"By risk class: { {k: f'${v/1e6:,.4f}M' for k, v in report.by_risk_class.items()} }")
print(f"Wall time: {elapsed:.1f}s")


# Diff: Native vs ParGraph

In [ ]:
import difflib
import json
import re

from IPython.display import HTML, display

with open("FRTB.ipynb") as _f:
    _nb = json.load(_f)
native_code = "".join(_nb["cells"][3]["source"])
parallel_code = "".join(_nb["cells"][9]["source"])

differ = difflib.HtmlDiff(wrapcolumn=90)
table = differ.make_table(
    native_code.splitlines(),
    parallel_code.splitlines(),
    fromdesc="Native Python",
    todesc="With ParGraph",
    context=False,
)

# Strip unwanted columns: navigation links, line numbers, and nowrap attribute
table = re.sub(r'<td class="diff_next"[^>]*>.*?</td>', "", table)
table = re.sub(r'<th class="diff_next"[^>]*>.*?</th>', "", table)
table = re.sub(r'<td class="diff_header"[^>]*>.*?</td>', "", table)
table = table.replace('colspan="2"', "")
table = table.replace(' nowrap="nowrap"', "")
table = re.sub(
    r"(\s*<colgroup></colgroup>){6}",
    "\n        <colgroup></colgroup> <colgroup></colgroup>",
    table,
)

css = """<style>
    table.diff { font-family: monospace; font-size: 10px; border-collapse: collapse; width: 100%; }
    table.diff td { padding: 2px 8px; white-space: pre-wrap; word-break: break-all; text-align: left !important; }
    table.diff th { padding: 6px 8px; text-align: center; background-color: #f0f0f0; font-size: 14px; }
    td.diff_add { background-color: #e6ffec; }
    td.diff_chg { background-color: #fff8c5; }
    td.diff_sub { background-color: #ffebe9; }
    span.diff_add { background-color: #ccffd8; }
    span.diff_chg { background-color: #fff2a8; }
    span.diff_sub { background-color: #ffc0c0; }
</style>"""

display(HTML(css + table))

Native Python,With ParGraph
import os,import os
,
"os.environ[""OPENBLAS_NUM_THREADS""] = ""1""","os.environ[""OPENBLAS_NUM_THREADS""] = ""1"""
"os.environ[""GOTO_NUM_THREADS""] = ""1""","os.environ[""GOTO_NUM_THREADS""] = ""1"""
"os.environ[""OMP_NUM_THREADS""] = ""1""","os.environ[""OMP_NUM_THREADS""] = ""1"""
"os.environ[""MKL_NUM_THREADS""] = ""1""","os.environ[""MKL_NUM_THREADS""] = ""1"""
,
import math,import math
import random,import random
import struct,import struct


# Statistics
## Sequential:
1 Core, walltime
```bash
real    12m1.000s
```

## ParFun (parallel):
16 Workers, walltime
```bash
real    0m58.900s
```
## ParFun Speedup:
`12.2x`

## ParGraph (parallel):
16 Workers, walltime
```bash
real    1m2.900s
```
## ParGraph Speedup:
`11.5x`